# FreshMart Lab 0: Environment Verification
**Microsoft Fabric Data Science**

ตรวจว่า workspace / Lakehouse FreshMart พร้อมก่อน Lab 1

1. เปิด workspace `labs` แล้วแนบ Default Lakehouse = `lh_freshmart`
2. รันเซลล์ด้านล่างตามลำดับ
3. ถ้าตาราง Bronze ยังไม่พร้อม ระบบอ่าน CSV จาก `Files/raw/` ให้อัตโนมัติ


In [ ]:
from pathlib import Path
import pandas as pd

def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None

def load_csv(file_name: str) -> pd.DataFrame:
    found = _first_existing([
        f"/lakehouse/default/Files/raw/{file_name}",
        f"Files/raw/{file_name}",
        f"../data/{file_name}",
        f"labs/data/{file_name}",
        file_name,
    ])
    if found is None:
        raise FileNotFoundError(f"Cannot find {file_name}. Upload it to Files/raw or place it under labs/data.")
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)

def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    try:
        frame = spark.read.table(table_name).toPandas()
        print(f"Loaded Spark table {table_name}: {len(frame):,} rows")
        return frame
    except Exception as exc:
        print(f"Spark table '{table_name}' unavailable ({exc}). Falling back to CSV.")
        return load_csv(file_name)


### โหลด Bronze (ตารางก่อน แล้วค่อย CSV)


In [ ]:
df_tx = load_table_or_csv("bronze_transactions", "freshmart_transactions.csv")
df_cust = load_table_or_csv("bronze_customers", "freshmart_customers.csv")

print(f"Transactions rows: {len(df_tx):,}")
print(f"Customers rows: {len(df_cust):,}")
print("Transaction columns:", list(df_tx.columns))
print("Customer columns:", list(df_cust.columns))
display(df_tx.head(5))
display(df_cust.head(5))


### จุดตรวจ (ผ่านแล้วค่อยเข้า Lab 1)


In [ ]:
if len(df_tx) != 3000:
    raise AssertionError(
        f"คาดว่าธุรกรรม 3,000 แถว แต่ได้ {len(df_tx):,} — ตรวจ Files/raw หรือตาราง bronze_transactions"
    )
if len(df_cust) != 1500:
    raise AssertionError(
        f"คาดว่าสมาชิก 1,500 แถว แต่ได้ {len(df_cust):,} — ตรวจ Files/raw หรือตาราง bronze_customers"
    )
print("Lab 0 verification passed")
